# Evaluation – Step 1: Generate Test Questions

This notebook samples documents from the indexed data and uses an LLM to generate
realistic test questions. The resulting questions are saved to `eval/questions.json`
and can be fed to the agent in `evaluations.ipynb`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'app'))

import json, random
from pathlib import Path
from pydantic import BaseModel
from pydantic_ai import Agent
from pydantic_ai.models.groq import GroqModel
import ingest

In [ ]:
# ── Load & index data ──────────────────────────────────────────────────────
index, docs = ingest.index_data(
    'evidentlyai', 'docs',
    chunk=True,
    chunking_params={'method': 'sliding_window', 'size': 2000, 'step': 1000},
)
print(f'Total chunks: {len(docs)}')

In [ ]:
# ── Question generator ─────────────────────────────────────────────────────
GENERATION_PROMPT = """
You help create test questions for an AI agent that answers questions
about the Evidently AI library.

Based on the provided documentation chunks, generate one realistic question
per chunk — questions a developer might actually ask.

Keep questions varied: some specific ('How do I …?'), some conceptual ('What is …?').
""".strip()

class QuestionsList(BaseModel):
    questions: list[str]

generator = Agent(
    name='question_generator',
    instructions=GENERATION_PROMPT,
    model=GroqModel('llama-3.3-70b-versatile'),
    output_type=QuestionsList,
)

In [ ]:
# ── Sample docs and generate questions ────────────────────────────────────
SAMPLE_SIZE = 20   # adjust as needed
sample = random.sample(docs, min(SAMPLE_SIZE, len(docs)))
prompt_input = json.dumps([{'content': d.get('content', '')[:800], 'filename': d.get('filename', '')} for d in sample])

result = await generator.run(prompt_input)
questions = result.output.questions
print(f'Generated {len(questions)} questions')
for q in questions:
    print(' •', q)

In [ ]:
# ── Save questions ─────────────────────────────────────────────────────────
out_path = Path('questions.json')
out_path.write_text(json.dumps(questions, indent=2))
print(f'Saved → {out_path}')